# Credit Card Customer Churn & Risk Segmentation Dashboard

### Question: Which credit card customer segments are most likely to churn, and what account and usage patterns can predict that risk?

### Stakeholder Framing: A bank's customer retention team requires a queryable tool that ranks currently active customers by churn risk so that retention offers can be targeted at the right customers before they leave.

In [1]:
%pip install pandas numpy sqlalchemy matplotlib seaborn flask

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# Project root = the folder this notebook lives in
PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\matte\Local\PortfolioFiles\Customer_Churn_Dashboard


In [4]:
# Create folder structure
for folder in ["data/raw", "db", "etl", "analysis", "app/templates", "app/static"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

raw_path = PROJECT_ROOT / "data/raw/BankChurners.csv"

if not raw_path.exists():
    raise FileNotFoundError(
        f"Expected the raw dataset at {raw_path}.\n"
        "Download BankChurners.csv from "
        "https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers "
        "and place it there before continuing."
    )

df_raw = pd.read_csv(raw_path)
print(f"Loaded {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

Loaded 10127 rows, 23 columns


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [5]:
# Drop Naive Bayes columns (dataset documentation says not to use)
nb_cols = [c for c in df_raw.columns if c.startswith("Naive_Bayes")]
print(f"Dropping {len(nb_cols)} Naive Bayes columns: {nb_cols}")

df = df_raw.drop(columns=nb_cols)
df.dtypes

Dropping 2 Naive Bayes columns: ['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']


CLIENTNUM                     int64
Attrition_Flag                  str
Customer_Age                  int64
Gender                          str
Dependent_count               int64
Education_Level                 str
Marital_Status                  str
Income_Category                 str
Card_Category                   str
Months_on_book                int64
Total_Relationship_Count      int64
Months_Inactive_12_mon        int64
Contacts_Count_12_mon         int64
Credit_Limit                float64
Total_Revolving_Bal           int64
Avg_Open_To_Buy             float64
Total_Amt_Chng_Q4_Q1        float64
Total_Trans_Amt               int64
Total_Trans_Ct                int64
Total_Ct_Chng_Q4_Q1         float64
Avg_Utilization_Ratio       float64
dtype: object

### Create database schema

In [6]:
%%writefile db/schema.sql
CREATE TABLE IF NOT EXISTS customers (
    client_num          INTEGER PRIMARY KEY,
    attrition_flag        TEXT,
    customer_age            INTEGER,
    gender                    TEXT,
    dependent_count            INTEGER,
    education_level              TEXT,
    marital_status                 TEXT,
    income_category                  TEXT,
    card_category                      TEXT
);

CREATE TABLE IF NOT EXISTS account_activity (
    client_num             INTEGER REFERENCES customers(client_num),
    months_on_book            INTEGER,
    total_relationship_count    INTEGER,
    credit_limit                   REAL,
    total_revolving_bal               REAL,
    avg_open_to_buy                     REAL,
    total_trans_amt                       REAL,
    total_trans_ct                          INTEGER,
    avg_utilization_ratio                     REAL
);

Writing db/schema.sql


In [7]:
# Create schema

db_path = PROJECT_ROOT / "db/churn.db"
engine = create_engine(f"sqlite:///{db_path}")

# Run schema.sql to create the table structure
schema_sql = (PROJECT_ROOT / "db/schema.sql").read_text()
with engine.begin() as conn:
    for statement in schema_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("Schema created.")

Schema created.


In [8]:
# Load data into SQL

customer_cols = ["CLIENTNUM", "Attrition_Flag", "Customer_Age", "Gender",
                  "Dependent_count", "Education_Level", "Marital_Status",
                  "Income_Category", "Card_Category"]
activity_cols = ["CLIENTNUM", "Months_on_book", "Total_Relationship_Count",
                  "Credit_Limit", "Total_Revolving_Bal", "Avg_Open_To_Buy",
                  "Total_Trans_Amt", "Total_Trans_Ct", "Avg_Utilization_Ratio"]

customers = df[customer_cols].rename(columns={
    "CLIENTNUM": "client_num", "Attrition_Flag": "attrition_flag",
    "Customer_Age": "customer_age", "Gender": "gender",
    "Dependent_count": "dependent_count", "Education_Level": "education_level",
    "Marital_Status": "marital_status", "Income_Category": "income_category",
    "Card_Category": "card_category"
})

activity = df[activity_cols].rename(columns={
    "CLIENTNUM": "client_num", "Months_on_book": "months_on_book",
    "Total_Relationship_Count": "total_relationship_count",
    "Credit_Limit": "credit_limit", "Total_Revolving_Bal": "total_revolving_bal",
    "Avg_Open_To_Buy": "avg_open_to_buy", "Total_Trans_Amt": "total_trans_amt",
    "Total_Trans_Ct": "total_trans_ct", "Avg_Utilization_Ratio": "avg_utilization_ratio"
})

customers.to_sql("customers", engine, if_exists="append", index=False)
activity.to_sql("account_activity", engine, if_exists="append", index=False)

print(f"Loaded {len(customers)} rows into customers")
print(f"Loaded {len(activity)} rows into account_activity")

Loaded 10127 rows into customers
Loaded 10127 rows into account_activity


### Feature engineering in SQL

In [9]:
%%writefile etl/create_view.sql
DROP VIEW IF EXISTS customer_risk_features;

CREATE VIEW customer_risk_features AS
SELECT
    c.client_num,
    c.attrition_flag,
    c.customer_age,
    c.income_category,
    c.card_category,
    a.months_on_book,
    a.credit_limit,
    a.total_trans_ct,
    a.total_trans_amt,
    a.avg_utilization_ratio,
    a.total_relationship_count
FROM customers c
JOIN account_activity a ON c.client_num = a.client_num;

Writing etl/create_view.sql


In [10]:
view_sql = (PROJECT_ROOT / "etl/create_view.sql").read_text()
with engine.begin() as conn:
    for statement in view_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

features = pd.read_sql("SELECT * FROM customer_risk_features", engine)
print(f"{len(features)} rows in the joined feature view")
features.head()

10127 rows in the joined feature view


,client_num,attrition_flag,customer_age,income_category,card_category,months_on_book,credit_limit,total_trans_ct,total_trans_amt,avg_utilization_ratio,total_relationship_count
0,768805383,Existing Customer,45,$60K - $80K,Blue,39,12691.0,42,1144.0,0.061,5
1,818770008,Existing Customer,49,Less than $40K,Blue,44,8256.0,33,1291.0,0.105,6
2,713982108,Existing Customer,51,$80K - $120K,Blue,36,3418.0,20,1887.0,0.000,4
3,769911858,Existing Customer,40,Less than $40K,Blue,34,3313.0,20,1171.0,0.760,3
4,709106358,Existing Customer,40,$60K - $80K,Blue,21,4716.0,28,816.0,0.000,5
